In [3]:
from io import BytesIO
from io import StringIO
import os
import pydicom
import struct
from pydicom.datadict import keyword_for_tag
from pydicom.tag import Tag

In [4]:
raw_path_incisive = r"\\imp-isp1.cle.ms.philips.com\IQ-Recons$\Seth\Raw Data\CT5300\20260203112451775\RawData\1.3.46.670589.61.128.7.2026020311140194229591979950003.rawmdu"


In [26]:
raw_path_iPatient = r"\\imp-isp1.cle.ms.philips.com\IQ-Recons$\Seth\Raw Data\S2000\I00"

In [ ]:
import io
import pydicom

MAX_BYTES =7028

with open(raw_path_incisive, "rb") as f:
    data = f.read(MAX_BYTES)

ds = pydicom.dcmread(io.BytesIO(data), force=True)
print(ds)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 218
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Raw Data Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.33.1.63880378330464876600001.5382259186384767053
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.33.103.11
(0002,0013) Implementation Version Name         SH: 'EBW4.8 for CT'
(0002,0016) Source Application Entity Title     AE: 'CTN3CLARA'
-------------------------------------------------
(0003,0010) Private Creator                     LO: 'ELSCINT1'
(0003,1001) [Offset List Structure]             OW: Array of 644 elements
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'RAW']
(0008,0016) SOP Class UID

In [ ]:
def readRawAndExtractDicom(rawPath: str):
    try:
        ds = pydicom.dcmread(rawPath, defer_size="1KB", force=True)
        
        if (0x01f7, 0x10cc) in ds:
            ChessDataSet = ds[0x01f710cc]
        else:
            print("Pixel Data (01F7,10cc) not found in DICOM dataset.")
            return
    
        ChessDataStructVersion = struct.unpack("@l", ChessDataSet.value[0:4])[0]
        SeriesInitStructSize = struct.unpack("@l", ChessDataSet.value[4:8])[0]
        NumDicoms = struct.unpack("@l", ChessDataSet.value[8:12])[0]

        ChessDataStructSize = 0
        if ChessDataStructVersion == 100:
            ChessDataStructSize = 111 * 4
        elif ChessDataStructVersion == 200:
            ChessDataStructSize = 416 * 4
        DICOMOffset = ChessDataStructSize + SeriesInitStructSize

        ds_new = pydicom.dcmread(BytesIO(ChessDataSet.value[DICOMOffset:]))
        return ds_new
    except Exception as e:
        print(f"Error reading DICOM from raw file: {e}")
        raise e

In [6]:
ds_new = readRawAndExtractDicom(raw_path)

Pixel Data (7FE0,0010) not found in DICOM dataset.


In [53]:
from pydicom.datadict import keyword_for_tag
from pydicom.tag import Tag

In [25]:
def find_first_dicm_offset(raw_path: str) -> int:
    with open(raw_path, "rb") as f:
        chunk_size = 1024 * 1024 # 1 MB
        overlap = b""
        pos = 0
        
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            
            data = overlap + chunk
            i = data.find(b"DICM")
            
            if i != -1:
                return pos + i - len(overlap)
            
            overlap = data[-3:]
            pos += chunk_size

    return -1

In [21]:
def find_all_dicm_offsets(path):
    offsets = []
    chunk_size = 1024 * 1024  # 1 MB

    with open(path, "rb") as f:
        overlap = b""
        pos = 0

        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break

            data = overlap + chunk

            i = 0
            while True:
                i = data.find(b"DICM", i)
                if i == -1:
                    break

                offsets.append(pos + i - len(overlap))
                i += 4

            # keep last 3 bytes for boundary overlap
            overlap = data[-3:]
            pos += len(chunk)

    return offsets

In [24]:
11179862/(1024*1024)

10.661947250366211

In [14]:
def verify_offset(path, offset):
    with open(path, "rb") as f:
        f.seek(offset)
        return f.read(4)

In [23]:
p =0x00AA9756
print(verify_offset(raw_path_incisive, 11179865-3))
print(verify_offset(raw_path_iPatient, 128))

b'DICM'
b'DICM'


In [10]:
with open(raw_path_incisive, "rb") as f:
    f.seek(0x00AA9756)
    print(f.read(4))

b'DICM'


In [22]:
find_all_dicm_offsets(raw_path_iPatient)

[128,
 22876133,
 90690761,
 117278581,
 232910915,
 253863891,
 318070033,
 352343421,
 399860691,
 428876873,
 576197645,
 636547721,
 740883891,
 834470755,
 838044841,
 868436517,
 940990923,
 954006937,
 1124613803,
 1162749675,
 1358669349,
 1377451379,
 1464412101,
 1579241117,
 1595066229,
 1687585291,
 1805182043,
 1893225513,
 1908392120,
 1909298489,
 2016762025,
 2528135335,
 2609806701,
 2651479801,
 2716523265,
 2731337585,
 3001624117,
 3165923257,
 3229624297,
 3266930857,
 3352329497,
 3360728787,
 3464258741,
 3544491433,
 3768521183,
 3795117109,
 3813249377]

In [ ]:
import struct

VALID_VR = {
    b'AE', b'AS', b'AT', b'CS', b'DA', b'DS', b'DT',
    b'FD', b'FL', b'IS', b'LO', b'LT', b'OB', b'OD',
    b'OF', b'OL', b'OW', b'PN', b'SH', b'SL', b'SQ',
    b'SS', b'ST', b'TM', b'UC', b'UI', b'UL', b'UN',
    b'UR', b'US', b'UT'
}

MAX_ELSCINT_GAP = 1* 1024 * 1024  # 1 MB

CHUNK_SIZE = 1024 * 1024  # 1 MB


def find_dicom_in_raw(path):

    with open(path, "rb") as f:

        dicm_offset = find_first_dicm_offset(path)

        if dicm_offset is None:
            raise ValueError("No DICM marker found")

        # Real DICOM starts 128 bytes before DICM
        start = max(0, dicm_offset - 128)

        f.seek(start)

        print(f"Starting scan at {start}")

        last_valid_pos = start
        last_elscint_pos = start

        while True:

            chunk_start = f.tell()

            chunk = f.read(CHUNK_SIZE)

            if not chunk:
                print("Reached EOF")
                break

            chunk_len = len(chunk)

            i = 0

            while i < chunk_len - 8:

                absolute_pos = chunk_start + i

                if chunk[i:i+8] == b'ELSCINT':

                    last_elscint_pos = absolute_pos
                    last_valid_pos = absolute_pos


                vr = chunk[i+4:i+6]

                if vr in VALID_VR:
                    last_valid_pos = absolute_pos

                if (absolute_pos - last_elscint_pos) > MAX_ELSCINT_GAP:

                    print(f"No ELSCINT detected for {MAX_ELSCINT_GAP} bytes.")
                    print(f"Stopping at {last_valid_pos}")

                    return (start, last_valid_pos)

                i += 1

        return (start, last_valid_pos)

In [ ]:
find_dicom_in_raw(raw_path_incisive)
#(11179734, 11180540)
#(11179734, 11182160)


Starting scan at 11179734
No ELSCINT detected for 1048576 bytes.
Stopping at 12227780
12228311 11179734 12227780


(11179734, 12227780)

In [56]:
1024**2

1048576

In [41]:
"11180560".encode()

b'11180560'

In [ ]:
find_dicom_in_raw(raw_path_iPatient)

Unreasonable length 1359433815 at position 159078096, stopping scan.


(0, 7028)

In [68]:
(1032928-7028)/1024**2

0.9783744812011719

In [ ]:
import io
import pydicom

MAX_BYTES =7028
start, end = find_dicom_in_raw(raw_path_incisive)
print(start, end)
with open(raw_path_incisive, "rb") as f:
    f.seek(start)
    data = f.read(end - start)

ds = pydicom.dcmread(io.BytesIO(data), force=True)
print(ds)

Starting scan at 11179734
No ELSCINT detected for 1048576 bytes.
Stopping at 12227780
11179734 12227780
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311

In [61]:
1024* 1024

1048576

In [57]:
"""
Need a better way to apply keyword_for_tag to the entire dataset. This is due to nested sequences holding tags that do not
get translated with this method
"""

new_dict = {keyword_for_tag(key): val for key, val in ds_new.to_json_dict().items()}

In [58]:
keyword_for_tag(0x00080005)

'SpecificCharacterSet'

In [59]:
ds_new.to_json_dict()

{'00080005': {'vr': 'CS', 'Value': ['ISO_IR 100']},
 '00080008': {'vr': 'CS',
  'Value': ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']},
 '00080016': {'vr': 'UI', 'Value': ['1.2.840.10008.5.1.4.1.1.2']},
 '00080018': {'vr': 'UI',
  'Value': ['1.3.46.670589.61.128.2.2026020311233339700030001']},
 '00080020': {'vr': 'DA', 'Value': ['20260203']},
 '00080021': {'vr': 'DA', 'Value': ['20260203']},
 '00080022': {'vr': 'DA', 'Value': ['20260203']},
 '00080030': {'vr': 'TM', 'Value': ['112000.393000']},
 '00080031': {'vr': 'TM', 'Value': ['112333.319000']},
 '00080032': {'vr': 'TM', 'Value': ['112346.587']},
 '00080050': {'vr': 'SH', 'Value': ['AccessionField']},
 '00080060': {'vr': 'CS', 'Value': ['CT']},
 '00080070': {'vr': 'LO', 'Value': ['Philips']},
 '00080080': {'vr': 'LO', 'Value': ['PMS Pinecrest']},
 '00080081': {'vr': 'ST', 'Value': ['100 Park Av']},
 '00080090': {'vr': 'PN', 'Value': [{'Alphabetic': 'ReferringPhysField'}]},
 '00081010': {'vr': 'SH', 'Value': ['BAY 2']},
 '00081030': {

In [60]:
keyword_to_tag = {keyword_for_tag(key) or Tag(key): Tag(key) for key, val in ds_new.to_json_dict().items()}

In [61]:
keyword_for_tag("")

''

In [62]:
keyword_to_tag

{'SpecificCharacterSet': (0008,0005),
 'ImageType': (0008,0008),
 'SOPClassUID': (0008,0016),
 'SOPInstanceUID': (0008,0018),
 'StudyDate': (0008,0020),
 'SeriesDate': (0008,0021),
 'AcquisitionDate': (0008,0022),
 'StudyTime': (0008,0030),
 'SeriesTime': (0008,0031),
 'AcquisitionTime': (0008,0032),
 'AccessionNumber': (0008,0050),
 'Modality': (0008,0060),
 'Manufacturer': (0008,0070),
 'InstitutionName': (0008,0080),
 'InstitutionAddress': (0008,0081),
 'ReferringPhysicianName': (0008,0090),
 'StationName': (0008,1010),
 'StudyDescription': (0008,1030),
 'SeriesDescription': (0008,103E),
 'InstitutionalDepartmentName': (0008,1040),
 'PerformingPhysicianName': (0008,1050),
 'NameOfPhysiciansReadingStudy': (0008,1060),
 'OperatorsName': (0008,1070),
 'ManufacturerModelName': (0008,1090),
 'ReferencedImageSequence': (0008,1140),
 'IrradiationEventUID': (0008,3010),
 'PatientName': (0010,0010),
 'PatientID': (0010,0020),
 'PatientBirthDate': (0010,0030),
 'PatientSex': (0010,0040),
 'Ot

In [63]:
keyword_to_tag

{'SpecificCharacterSet': (0008,0005),
 'ImageType': (0008,0008),
 'SOPClassUID': (0008,0016),
 'SOPInstanceUID': (0008,0018),
 'StudyDate': (0008,0020),
 'SeriesDate': (0008,0021),
 'AcquisitionDate': (0008,0022),
 'StudyTime': (0008,0030),
 'SeriesTime': (0008,0031),
 'AcquisitionTime': (0008,0032),
 'AccessionNumber': (0008,0050),
 'Modality': (0008,0060),
 'Manufacturer': (0008,0070),
 'InstitutionName': (0008,0080),
 'InstitutionAddress': (0008,0081),
 'ReferringPhysicianName': (0008,0090),
 'StationName': (0008,1010),
 'StudyDescription': (0008,1030),
 'SeriesDescription': (0008,103E),
 'InstitutionalDepartmentName': (0008,1040),
 'PerformingPhysicianName': (0008,1050),
 'NameOfPhysiciansReadingStudy': (0008,1060),
 'OperatorsName': (0008,1070),
 'ManufacturerModelName': (0008,1090),
 'ReferencedImageSequence': (0008,1140),
 'IrradiationEventUID': (0008,3010),
 'PatientName': (0010,0010),
 'PatientID': (0010,0020),
 'PatientBirthDate': (0010,0030),
 'PatientSex': (0010,0040),
 'Ot

In [64]:
def ds_to_keyword_dict(ds):
    result = {}

    for elem in ds:
        if elem.VR == "SQ":  # handle sequences recursively
            result[elem.keyword or str(elem.tag)] = [
                ds_to_keyword_dict(item) for item in elem.value
            ]
        else:
            key = elem.keyword or str(elem.tag)  # fallback if no keyword
            result[key] = elem.value

    return result

def viewDicomData(ds):
    return ds_to_keyword_dict(ds)

In [65]:
viewDicomData(ds_new)

{'SpecificCharacterSet': 'ISO_IR 100',
 'ImageType': ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL'],
 'SOPClassUID': '1.2.840.10008.5.1.4.1.1.2',
 'SOPInstanceUID': '1.3.46.670589.61.128.2.2026020311233339700030001',
 'StudyDate': '20260203',
 'SeriesDate': '20260203',
 'AcquisitionDate': '20260203',
 'StudyTime': '112000.393000',
 'SeriesTime': '112333.319000',
 'AcquisitionTime': '112346.587',
 'AccessionNumber': 'AccessionField',
 'Modality': 'CT',
 'Manufacturer': 'Philips',
 'InstitutionName': 'PMS Pinecrest',
 'InstitutionAddress': '100 Park Av',
 'ReferringPhysicianName': 'ReferringPhysField',
 'StationName': 'BAY 2',
 'StudyDescription': 'Test Cardiac Scan',
 'SeriesDescription': '',
 'InstitutionalDepartmentName': 'CT#',
 'PerformingPhysicianName': 'PerformingPhysField',
 'NameOfPhysiciansReadingStudy': 'ReadingPhysField',
 'OperatorsName': 'OperatrField',
 'ManufacturerModelName': 'Incisive CT',
 'ReferencedImageSequence': [{'ReferencedSOPClassUID': '1.2.840.10008.5.1.4.1.1.2',


In [ ]:
"""
Structure of values:

Primary:
    'SpecificCharacterSet': {'vr': 'CS', 'Value': ['ISO_IR 100']}
Array of Primary:
    'ImageType': {'vr': 'CS','Value': ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']},
Person Name
    'ReferringPhysicianName': {'vr': 'PN','Value': [{'Alphabetic': 'ReferringPhysField'}]},

Nested Structure:
    'ReferencedImageSequence': {'vr': 'SQ',
  'Value': [
  {'00081150': {'vr': 'UI', 'Value': ['1.2.840.10008.5.1.4.1.1.2']},
    '00081155': {'vr': 'UI','Value': ['1.3.46.670589.61.128.2.20260203112104777010100011482450593']}},
   {'00081150': {'vr': 'UI', 'Value': ['1.2.840.10008.5.1.4.1.1.2']},
    '00081155': {'vr': 'UI','Value': ['1.3.46.670589.61.128.2.20260203112131526010100023842926342']}}]},

    
Rules: Given Path to value, edit accordingly
Path:
    Primary: Key.0
    Array of Primary: Key.0, Key.1, etc.
    Person Name: Key.0.Alphabetic
    Nested Structure: Key.0.Key.0, Key.1.Key.0, etc (Note it can be even more nested than this)
"""

In [66]:
from pydicom.valuerep import PersonName
from datetime import datetime

def validate_vr(vr, value):
    try:
        if vr in ["LO", "SH", "ST", "LT", "UT", "CS", "AE"]:
            return str(value)

        elif vr == "PN":
            return PersonName(str(value))

        elif vr in ["DA"]:  # Date YYYYMMDD
            if isinstance(value, str):
                datetime.strptime(value, "%Y%m%d")
                return value
            return None

        elif vr in ["TM"]:  # Time HHMMSS
            if isinstance(value, str):
                datetime.strptime(value.split(".")[0], "%H%M%S")
                return value
            return None

        elif vr in ["UI"]:  # UID
            return str(value)

        elif vr in ["IS"]:  # Integer String
            return str(int(value))

        elif vr in ["DS"]:  # Decimal String
            return str(float(value))

        elif vr in ["US", "UL", "SS", "SL"]:
            return int(value)

        elif vr in ["FL", "FD"]:
            return float(value)

        elif vr in ["OB", "OW", "UN"]:
            if isinstance(value, (bytes, bytearray)):
                return value
            return None

        elif vr == "SQ":
            # Sequences must be list of datasets
            return value if isinstance(value, list) else None

        else:
            # fallback: allow but cast to string
            return str(value)

    except Exception:
        return None

In [149]:
def editDicomData(ds,path:str,new_value:str|int|float):
    valid_keys = set(elem.keyword if elem.keyword else elem.tag for elem in ds_new)
    keys = path.split(".")
    current_level = ds
    vr = None
    
    # Exclude last key
    for key in keys[:-1]:
        tagKey = keyword_to_tag.get(key, key)
        if type(tagKey) == str and tagKey.isdigit():
            current_level = current_level.value[int(tagKey)]
        else:
            # tagKey is of type Tag
            current_level = current_level[tagKey]
            vr = current_level.VR or vr
    
     # Check if the last key is a subkey (like 'Alphabetic' in a Person Name)
    last_key = keyword_to_tag.get(keys[-1], keys[-1])
    

    if type(last_key) == str and last_key.isdigit():
        validated_value = validate_vr(vr, new_value)
        if validated_value is None:
            return
        current_level.value[int(last_key)] = validated_value
    else:
        # last_key is of type Tag
        vr = current_level[last_key].VR or vr
        validated_value = validate_vr(vr, new_value)
        if validated_value is None:
            return
        current_level[last_key].value = validated_value



In [153]:
keyword_to_tag["PatientName"]

(0010,0010)

In [150]:
ds_new

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 193'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY_MODIFIED', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date                          DA: '20260203'
(0008,0021) Seri

In [ ]:
"""

For Editing, we note these facts
1. Primary values:  'InstitutionAddress': {'vr': 'ST', 'Value': ['100 Park Av']},
    To edit them - ds[key].value = new_value
2. Arrays of Primary values: 'ImageType': {'vr': 'CS','Value': ['ORIGINAL', 'PRIMARY_MODIFIED', 'AXIAL', 'HELICAL']},
    To edit them - ds[key].value[idx] = new_value # 0 <= idx < len(ds[key].value)
3. Nested Sequences: 'ReferencedImageSequence':
                        {'vr': 'SQ',  'Value': [
                            {'00081150': {'vr': 'UI', 'Value': ['1.2.840.10008.5.1.4.1.1.2']},'00081155': {'vr': 'UI','Value': ['1.3.46.670589.61.128.2.20260203112104777010100011482450593']}},
                        {'00081150': {'vr': 'UI','Value': ['1.2.840.10008.5.1.4.1.1.2']}, '00081155': {'vr': 'UI','Value': ['1.3.46.670589.61.128.2.20260203112131526010100023842926342']}}
                        ]},
    To edit them - ds[key1].value[idx][key2].value.* = new_value # 0 <= idx < len(ds[key1].value), 
"""

In [120]:
ds_new["PatientName"].VR

'PN'

In [104]:
ds_new["ReferencedImageSequence"].value[0]["00081155"].value

'1.3.46.670589.61.128.2.20260203112104777010100011482450593'

In [97]:
ds_new["PatientName"].value

'LastName123^FirstName456^^^'

In [99]:
ds_new["InstitutionAddress"].value

'100 Park Av'

In [77]:
ds_new["PatientName"]

(0010,0010) Patient's Name                      PN: 'LastName123^FirstName456^^^'

In [147]:
ds_new.save_as(r"C:\Users\320308966\Documents\RawDataManager\edited_dicom.dcm")

c:\Users\320308966\AppData\Local\anaconda3\Lib\site-packages\pydicom\charset.py:738: UserWarning: Unknown encoding 'ISO_IR 193' - using default encoding instead
  _warn_about_invalid_encoding(encoding)


In [111]:
editDicomData(ds_new, "PatientName", 'LastName123^FirstName456^^^')
#'LastName123^FirstName456^^^'

Current level before editing: Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 193'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date                          DA: '20260

In [114]:
ds_new["ImageType"]

(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY_MODIFIED', 'AXIAL', 'HELICAL']

In [113]:
editDicomData(ds_new, "ImageType.1", "PRIMARY_MODIFIED")

Current level after accessing ImageType: (0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
Current level before editing: (0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
VR at current level: CS


In [ ]:
ds_new

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 193'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY_MODIFIED', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date                          DA: '20260203'
(0008,0021) Seri

In [115]:
editDicomData(ds_new, "ReferringPhysicianName", "ReferringPhysField_MODIFIED")

Current level before editing: Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 193'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY_MODIFIED', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date                          D

In [118]:
editDicomData(ds_new, "ReferencedImageSequence.0.ReferencedSOPClassUID", "1.2.840.10008.5.1.4.1.1.2")

Current level after accessing ReferencedImageSequence: (0008,1140) Referenced Image Sequence           SQ: <Sequence, length 2>
Current level before editing: (0008,1150) Referenced SOP Class UID            UI: NEW_UID
(0008,1155) Referenced SOP Instance UID         UI: 1.3.46.670589.61.128.2.20260203112104777010100011482450593
VR at current level: SQ
VR at current level: UI


In [159]:
ds_new["ReferencedImageSequence"][0]["ReferencedSOPClassUID"]

(0008,1150) Referenced SOP Class UID            UI: CT Image Storage

In [160]:
ds_new["ReferencedImageSequence"][0][0x00081150]

(0008,1150) Referenced SOP Class UID            UI: CT Image Storage

In [158]:
keyword_to_tag["ReferencedSOPClassUID"]

KeyError: 'ReferencedSOPClassUID'

In [154]:
ds_new["ReferencedImageSequence"]

(0008,1140) Referenced Image Sequence           SQ: <Sequence, length 2>

In [189]:
import RawDicomManager

In [197]:
dcm_manager = RawDicomManager(raw_path)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date                          DA: '20260203'
(0008,0021) Series Date  

In [198]:
dcm_manager.editDicomData("PatientName", 'NewPatientName')

In [199]:
dcm_manager.ds["PatientName"]

(0010,0010) Patient's Name                      PN: 'NewPatientName'

In [200]:
dcm_manager.saveAsRawData("./edited_raw.rawmdu")

In [201]:
dcm_manager_1 = RawDicomManager("./edited_raw.rawmdu")

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0000,0000) Command Group Length                UL: None
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date         

In [205]:
print(dcm_manager_1.ds)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 154
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.46.670589.61.128.2
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.46.670589.54.2.21.7
(0002,0013) Implementation Version Name         SH: '21.7.0.0'
-------------------------------------------------
(0000,0000) Command Group Length                UL: None
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELICAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.46.670589.61.128.2.2026020311233339700030001
(0008,0020) Study Date         

In [ ]:
"""
Steps to wrap this applicaton
1) User opens the application and selects a raw file
2) Application reads the raw file, extracts the DICOM dataset, and displays it in a human-readable format 
    (using keywords instead of tags).
3) User can select a specific value (or a list of values using a json file) to edit by providing a path (e.g., "ImageType.1" for the second value in the ImageType array, or "ReferencedImageSequence.0.ReferencedSOPClassUID" for a nested sequence value).
4) Application validates the new value against the VR of the DICOM element and updates it if valid, otherwise rejects the change and informs the user of the validation error.

"""

In [125]:
import pydicom
from pydicom.dataset import Dataset
from pydicom.sequence import Sequence

def create_test_ds():
    ds = Dataset()

    # 1. Primary value
    ds.InstitutionAddress = "100 Park Av"  # ST

    # 2. Array of values
    ds.ImageType = ["ORIGINAL", "PRIMARY", "AXIAL", "HELICAL"]  # CS

    # 3. Sequence
    seq_item1 = Dataset()
    seq_item1.ReferencedSOPClassUID = "1.2.3"
    seq_item1.ReferencedSOPInstanceUID = "1.2.3.4"

    seq_item2 = Dataset()
    seq_item2.ReferencedSOPClassUID = "4.5.6"
    seq_item2.ReferencedSOPInstanceUID = "4.5.6.7"

    ds.ReferencedImageSequence = Sequence([seq_item1, seq_item2])

    return ds

In [136]:
def test_edit_primary_value():
    ds = create_test_ds()

    editDicomData(ds, "InstitutionAddress", "200 Main St")

    assert ds.InstitutionAddress == "200 Main St"

try:
    test_edit_primary_value()
    print("Success")
except AssertionError:
    print("Failed")    


Success


In [132]:
def test_edit_primary_invalid_vr():
    ds = create_test_ds()

    # ST expects string → passing int should still coerce to string (valid)
    editDicomData(ds, "InstitutionAddress", 123)

    assert ds.InstitutionAddress == "123"
try:
    test_edit_primary_invalid_vr()
    print("Success")
except AssertionError:
    print("Failed")    

Success


In [137]:
def test_edit_array_index():
    ds = create_test_ds()

    editDicomData(ds, "ImageType.1", "PRIMARY_MODIFIED")

    assert ds.ImageType[1] == "PRIMARY_MODIFIED"

try:
    test_edit_array_index()
    print("Success")
except AssertionError:
    print("Failed")   

Success


In [138]:
def test_edit_array_out_of_bounds():
    ds = create_test_ds()

    try:
        editDicomData(ds, "ImageType.10", "INVALID")
        assert False, "Should have failed"
    except Exception:
        assert True


try:
    test_edit_array_out_of_bounds()
    print("Success")
except AssertionError:
    print("Failed")    

Success


In [139]:
def test_edit_array_invalid_vr():
    ds = create_test_ds()

    # CS = Code String → should not accept spaces ideally
    editDicomData(ds, "ImageType.1", "INVALID VALUE")

    # Should remain unchanged if validation is strict
    assert ds.ImageType[1] == "PRIMARY"
try:
    test_edit_array_invalid_vr()
    print("Success")
except AssertionError:
    print("Failed")  

Failed


In [140]:
def test_edit_sequence_value():
    ds = create_test_ds()

    editDicomData(
        ds,
        "ReferencedImageSequence.0.ReferencedSOPClassUID",
        "1.2.840.10008.5.1.4.1.1.2"
    )

    assert ds.ReferencedImageSequence[0].ReferencedSOPClassUID == \
        "1.2.840.10008.5.1.4.1.1.2"

try:
    test_edit_sequence_value()
    print("Success")
except AssertionError:
    print("Failed")  

Success


In [141]:
def test_edit_sequence_invalid_uid():
    ds = create_test_ds()

    editDicomData(
        ds,
        "ReferencedImageSequence.0.ReferencedSOPClassUID",
        "INVALID_UID"
    )

    # Should not update
    assert ds.ReferencedImageSequence[0].ReferencedSOPClassUID == "1.2.3"

try:
    test_edit_sequence_invalid_uid()
    print("Success")
except AssertionError:
    print("Failed")  

Failed


c:\Users\320308966\AppData\Local\anaconda3\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: 'INVALID_UID'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


In [142]:
def test_invalid_path():
    ds = create_test_ds()

    try:
        editDicomData(ds, "NonExistentTag", "value")
        assert False, "Should fail"
    except Exception:
        assert True

try:
    test_invalid_path()
    print("Success")
except AssertionError:
    print("Failed")  

Success


In [143]:
def test_invalid_sequence_index():
    ds = create_test_ds()

    try:
        editDicomData(
            ds,
            "ReferencedImageSequence.5.ReferencedSOPClassUID",
            "1.2.3"
        )
        assert False, "Should fail"
    except Exception:
        assert True
    
try:
    test_invalid_sequence_index()
    print("Success")
except AssertionError:
    print("Failed")  

Success


In [144]:
def test_navigation_integrity():
    ds = create_test_ds()

    editDicomData(
        ds,
        "ReferencedImageSequence.1.ReferencedSOPInstanceUID",
        "9.9.9"
    )

    assert ds.ReferencedImageSequence[1].ReferencedSOPInstanceUID == "9.9.9"
try:
    test_navigation_integrity()
    print("Success")
except AssertionError:
    print("Failed")  

Success


In [145]:
def test_vr_propagation_bug():
    ds = create_test_ds()

    editDicomData(
        ds,
        "ReferencedImageSequence.0.ReferencedSOPClassUID",
        123  # invalid for UI
    )

    # Should NOT update if VR is correctly resolved
    assert ds.ReferencedImageSequence[0].ReferencedSOPClassUID == "1.2.3"
try:
    test_vr_propagation_bug()
    print("Success")
except AssertionError:
    print("Failed")  

Failed
